# Suite TSDEF — TypeScript definitions

The names a TypeScript or JavaScript program declares, in the scopes ECMAScript and TypeScript give them
(`TypeScript/Definitions.define`): one entity per name and kind per scope, merging what TypeScript merges, and
lookup outward that keeps values, types and namespaces apart. Each case checks the entities, their declarations,
and what names resolve to.

In [ ]:
import { readFileSync } from "node:fs";

import { Definitions as FD, Syntax as F } from "@mbse/programs/Framework";
import { Definitions as D, Syntax as S, TypeScript59 } from "@mbse/programs/TypeScript";

import { assert, equal } from "./support.js";

const TS = TypeScript59.STANDARD;
const SOURCES = "../../conformance/sources";
type Standard = { parse(text: string): S.Program };

function define(text: string, standard: Standard = TS): [S.Program, D.TypeScriptProgram] {
  const tree = standard.parse(text);
  return [tree, D.define(tree)];
}

const entities = (program: FD.Program) => [...program.entities()];
const listing = (program: FD.Program) => entities(program).map((e) => [e.kind, e.qualified_name(), e.declarations.length]);
/** The identifiers of `tree` named `name`, in source order. */
const names = (tree: F.Node, name: string): any[] => [...F.walk(tree)].filter((n) =>
  (n instanceof S.Identifier || n instanceof S.JSXIdentifier) && n.name === name);
const refers = (program: D.TypeScriptProgram, node: any) =>
  D.referents(program, node).map((e) => `${e.kind}:${e.qualified_name()}`);
const named = (program: FD.Program, name: string) => entities(program).find((e) => e.name === name) as FD.Entity;

## TSDEF-01 · What declarations declare, and which merge: one entity per name and kind per scope

In [ ]:
let [tree, program] = define(`
var v = 1; var v;
let [a, , ...b] = c, { d, e: f = 1, ...g } = h;
const k = 1;
function over(x: string): string;
function over(x: any) { return x; }
interface I { a: string }
interface I { b: number }
namespace N { export const x = 1; }
namespace N { export const y = 2; }
enum E { A }
enum E { B = 1 }
type T = string;
class C {}
import def, { one as two, type three } from "m";
import * as all from "m";
import eq = require("m");
declare function amb(): void;
export default function () {}
`);
equal(listing(program), [
  ["variable", "v", 2], ["variable", "a", 1], ["variable", "b", 1], ["variable", "d", 1], ["variable", "f", 1],
  ["variable", "g", 1], ["variable", "k", 1], ["function", "over", 2], ["parameter", "over.x", 1],
  ["parameter", "over.x", 1], ["interface", "I", 2], ["property", "I.a", 1], ["property", "I.b", 1],
  ["namespace", "N", 2], ["variable", "N.x", 1], ["variable", "N.y", 1], ["enum", "E", 2],
  ["enum member", "E.A", 1], ["enum member", "E.B", 1], ["type alias", "T", 1], ["class", "C", 1],
  ["import", "def", 1], ["import", "two", 1], ["import", "three", 1], ["import", "all", 1], ["import", "eq", 1],
  ["function", "amb", 1]]);
const over = named(program, "over");
equal(over.declarations.map((d) => (d as F.Node).kind().KIND), ["TSDeclareFunction", "FunctionDeclaration"]);
assert(over.definition instanceof S.FunctionDeclaration);
assert(over.scope?.node === over.definition); // an overloaded function's scope is its implementation's
const amb = named(program, "amb");
assert(amb.definition === null && amb.scope !== null);
const iface = entities(program).find((e) => e.kind === "interface") as FD.Entity;
assert(iface.definition === iface.declarations[0] && iface.scope?.kind === "interface");
assert((program.lookup("I.a")[0] as FD.Entity).definition === null); // a signature declares, and defines nothing
assert(program.root.separator === "." && (program.lookup("N.y")[0] as FD.Entity).kind === "variable");
equal(program.lookup("E").map((e) => e.kind), ["enum"]);
assert((program.lookup("E.B")[0] as FD.Entity).parent?.kind === "enum");
console.log(listing(program).length, "entities");

## TSDEF-02 · Scopes: `var` in its function, the rest in their blocks; loops, switches, catches and static blocks

In [ ]:
[tree, program] = define(`
function f(p: number) {
    if (p) { var hoisted = 1; let local = 2; }
    for (let i = 0; i < p; i++) { const j = i; }
    for (const key in o) {}
    for (const value of o) {}
    switch (p) { case 1: let s = 1; }
    try {} catch ({ message }) { message; }
    label: for (;;) { break label; }
    return hoisted + i;
}
class K { static { var inStatic = 1; } }
const fe = function named() { return named; };
const arrow = (a = b, { c } = a) => a + c;
const ce = class Named { m() { return Named; } };
const { [key]: renamed } = o;
const anonymous = class {};
`);
equal(listing(program), [
  ["function", "f", 1], ["parameter", "f.p", 1], ["variable", "f.hoisted", 1], ["variable", "local", 1],
  ["variable", "i", 1], ["variable", "j", 1], ["variable", "key", 1], ["variable", "value", 1],
  ["variable", "s", 1], ["variable", "message", 1], ["class", "K", 1], ["variable", "inStatic", 1],
  ["variable", "fe", 1], ["function", "named", 1], ["variable", "arrow", 1], ["parameter", "a", 1],
  ["parameter", "c", 1], ["variable", "ce", 1], ["class", "Named", 1], ["method", "Named.m", 1],
  ["variable", "renamed", 1], ["variable", "anonymous", 1]]);
const key = names(tree, "key").at(-1); // the pattern's computed key, outside the loop whose `key` it is
assert(program.scope_of(key) === program.root);
equal(refers(program, key), []);
const scopes = new Map(entities(program).map((e) => [e.name, e.parent?.kind]));
equal(program.lookup("Named"), []);
assert(scopes.get("Named") === "block"); // a class expression's name is its own
equal(["local", "i", "s", "message", "inStatic"].map((name) => scopes.get(name)),
  ["block", "block", "block", "block", "static block"]);
equal(refers(program, names(tree, "hoisted").at(-1)), ["variable:f.hoisted"]);
equal(refers(program, names(tree, "i").at(-1)), []); // `i` is the loop's
equal(refers(program, names(tree, "named").at(-1)), ["function:named"]);
equal(refers(program, names(tree, "Named").at(-1)), ["class:Named"]);
equal(refers(program, names(tree, "b")[0]), []);
equal(refers(program, names(tree, "a").at(-1)), ["parameter:a"]);
assert(program.scope_of(names(tree, "label")[1]) === null); // a label names no entity
equal(refers(program, names(tree, "message").at(-1)), ["variable:message"]);
console.log("ok");

## TSDEF-03 · Classes and interfaces: members, accessors, private names, parameter properties and heritage

In [ ]:
[tree, program] = define(`
abstract class Box<T> extends Base<T> implements Shape {
    #secret = 1;
    static count = 0;
    accessor size = 1;
    [computed] = 2;
    'quoted' = 3;
    "double" = 4;
    0 = 5;
    constructor(private readonly value: T, public label = "x", plain: number) { super(); }
    get area(): number { return this.#secret; }
    set area(v: number) {}
    abstract run(): void;
    method?(): void;
    method?(): void {}
    [key: string]: unknown;
    @decorated field = Box.count;
}
interface Shape<T = number> extends Base<T> {
    area: number; move(): void; get size(): number; [k: string]: unknown; [Symbol.iterator](): void
}
`);
const box = named(program, "Box").scope as FD.Scope;
equal(box.entities().map((e) => [e.kind, e.name, e.declarations.length]), [
  ["property", "#secret", 1], ["property", "count", 1], ["accessor", "size", 1], ["property", "quoted", 1],
  ["property", "double", 1], ["property", "0", 1], ["method", "constructor", 1], ["property", "value", 1],
  ["property", "label", 1], ["accessor", "area", 2],
  ["method", "run", 1], ["method", "method", 2], ["property", "field", 1]]);
assert(box.parent?.kind === "type parameters");
equal(refers(program, names(tree, "T")[1]), ["type parameter:T"]); // `Base<T>`
equal(refers(program, names(tree, "Base")[0]), []);
equal(refers(program, names(tree, "Shape")[0]), ["interface:Shape"]);
equal(refers(program, names(tree, "computed")[0]), []);
equal(refers(program, names(tree, "count")[1]), []); // a property's name depends on the value
equal(refers(program, names(tree, "Box")[1]), ["class:Box"]);
const constructor = box.entities().find((e) => e.name === "constructor") as FD.Entity;
equal(constructor.scope?.entities().map((e) => e.kind), ["parameter", "parameter", "parameter"]);
const method = box.entities().find((e) => e.name === "method") as FD.Entity;
assert((method.definition as any).value instanceof S.FunctionExpression);
const shape = entities(program).find((e) => e.kind === "interface")?.scope as FD.Scope;
equal(shape.entities().map((e) => [e.kind, e.name]), [["property", "area"], ["method", "move"], ["accessor", "size"]]);
assert((program.lookup("Box.count")[0] as FD.Entity).kind === "property");
console.log("ok");

## TSDEF-04 · Meanings: a name is a value, a type or a namespace where it is written, and finds only those

In [ ]:
[tree, program] = define(`
type Point = { x: number };
const Point = 1;
namespace Geo { export interface Shape {} }
let p: Point = Point;
let q: typeof Point;
let s: Geo.Shape;
let g = Geo;
let a = <Point>p, b = p as Point, c = p satisfies Point, d = [p] as const;
function guard(x: unknown): x is Point { return true; }
export { Point };
export { Point as Other } from "m";
const Point2 = Point;
`);
const point = names(tree, "Point");
equal(point.map((n) => program.space_of(n)),
  [null, null, "type", "value", "value", "type", "type", "type", "type", null, null, null, "value"]);
equal(refers(program, point[2]), ["type alias:Point"]);
equal(refers(program, point[3]), ["variable:Point"]);
equal(refers(program, point[9]), ["type alias:Point", "variable:Point"]); // an export names whatever it means
assert(program.scope_of(point[10]) === null && program.scope_of(point[11]) === null); // exported, or from elsewhere
const geo = names(tree, "Geo");
equal(geo.map((n) => program.space_of(n)), [null, "namespace", "value"]);
equal(refers(program, geo[1]), ["namespace:Geo"]);
equal(refers(program, names(tree, "Shape")[1]), []);
equal(refers(program, names(tree, "x")[1]), ["parameter:guard.x"]);
assert(program.scope_of(names(tree, "const")[0]) === null);
equal(D.MEANINGS.class, new Set(["value", "type", "namespace"]));
equal(D.MEANINGS.interface, new Set(["type"]));
console.log("ok");

## TSDEF-05 · Namespaces, ambient modules, `declare global` and `import a = b.c`

In [ ]:
[tree, program] = define(`
namespace Outer.Inner { export const deep = 1; }
namespace Outer { export namespace Inner { export const also = 2; } }
declare module "virtual" { export const v: number; }
declare module "short";
declare module "virtual" { export const w: number; }
declare global { interface Window { app: string } }
import Alias = Outer.Inner;
import Missing = Nowhere.Thing;
import Self = Self.X;
import Required = require("m");
const use = Alias.deep;
`);
equal(listing(program).slice(0, 6), [
  ["namespace", "Outer", 2], ["namespace", "Outer.Inner", 2], ["variable", "Outer.Inner.deep", 1],
  ["variable", "Outer.Inner.also", 1], ["module", "virtual", 2], ["variable", "virtual.v", 1]]);
equal(program.lookup("short").map((e) => e.kind), ["module"]);
assert((program.lookup("short")[0] as FD.Entity).scope?.kind === "module");
assert((program.lookup("Window")[0] as FD.Entity).kind === "interface");
const [alias, missing, self, required] = ["Alias", "Missing", "Self", "Required"].map((name) =>
  program.lookup(name)[0] as FD.Entity);
assert(alias?.target?.qualified_name() === "Outer.Inner" && alias.resolved().kind === "namespace");
assert(missing?.target === null && self?.target === null && required?.target === null);
equal(refers(program, names(tree, "Alias")[1]), ["import:Alias"]);
console.log("ok");

## TSDEF-06 · Types' own scopes: signatures, mapped types, `infer`, and JSX's components

In [ ]:
[tree, program] = define(`
enum Flags { A, // a comment
  [computed] = 1 }
type Mapped<T> = { [K in keyof T]: T[K] };
type Element<T> = T extends (infer E extends string)[] ? E : never;
type Fn = <U>(arg: U, ...rest: U[]) => U;
type Ctor = new (x: number) => object;
type Lit = { (a: string): void; new (b: string): Lit; m<V>(c: V): V; [k: string]: unknown };
`);
equal(entities(program).map((e) => [e.kind, e.name, e.parent?.kind]), [
  ["enum", "Flags", "module"], ["enum member", "A", "enum"], ["type alias", "Mapped", "module"],
  ["type parameter", "T", "type parameters"], ["type parameter", "K", "type parameters"],
  ["type alias", "Element", "module"], ["type parameter", "T", "type parameters"],
  ["type parameter", "E", "type parameters"], ["type alias", "Fn", "module"], ["type parameter", "U", "function"],
  ["parameter", "arg", "function"], ["parameter", "rest", "function"], ["type alias", "Ctor", "module"],
  ["parameter", "x", "function"], ["type alias", "Lit", "module"], ["parameter", "a", "function"],
  ["parameter", "b", "function"], ["type parameter", "V", "function"], ["parameter", "c", "function"]]);
equal(refers(program, names(tree, "V").at(-1)), ["type parameter:V"]);
equal(refers(program, names(tree, "K")[1]), ["type parameter:K"]);
equal(refers(program, names(tree, "E")[1]), ["type parameter:E"]);
equal(refers(program, names(tree, "U").at(-1)), ["type parameter:U"]);
equal(refers(program, names(tree, "Lit")[1]), ["type alias:Lit"]);
assert(program.scope_of(names(tree, "k")[0]) === null); // an index signature's parameter names nothing
[tree, program] = define(`
function Component() {}
const ns = { Item: Component };
const view = <div><Component /><ns.Item>text</ns.Item><svg:path d={ns} /></div>;
`, TypeScript59.JSX);
const component = names(tree, "Component");
equal(refers(program, names(tree, "ns").at(-1)), ["variable:ns"]);
equal(refers(program, component[2]), ["function:Component"]);
assert(program.space_of(component[2]) === "value");
equal(refers(program, names(tree, "ns")[1]), ["variable:ns"]);
assert(program.scope_of(names(tree, "Item")[1]) === null);
assert(program.scope_of(names(tree, "div")[0]) === null && program.scope_of(names(tree, "svg")[0]) === null);
console.log("ok");

## TSDEF-07 · Every entity of the conformance source, and every node located but the names of properties

In [ ]:
[tree, program] = define(readFileSync(`${SOURCES}/typescript59.ts`, "utf8"));
const counts = new Map<string, number>();
for (const entity of program.entities()) counts.set(entity.kind, (counts.get(entity.kind) ?? 0) + 1);
equal(new Set(counts.keys()), new Set(Object.keys(D.MEANINGS)));
equal(new Set([...F.walk(tree)].filter((n) => program.scope_of(n) === null).map((n) => n.kind().KIND)),
  new Set(["Identifier", "PrivateIdentifier"]));
const point2 = program.lookup("Point2")[0] as FD.Entity;
assert(point2.kind === "import" && point2.target?.qualified_name() === "Geometry.Point");
console.log([...counts.values()].reduce((x, y) => x + y), "entities", [...counts].sort());